In [ ]:
import pandas as pd
import joblib
import re
from urllib.parse import urlparse
import os

# 1. Load the model
model = joblib.load('csec_trained_model.pkl')
expected_features = list(model.feature_names_in_)

def get_counts(text, is_missing=False):
    """If the URL part is missing, return -1 for all counts to match the CSV"""
    if is_missing:
        return {char: -1 for char in ['dot', 'hyphen', 'underline', 'slash', 'questionmark', 'equal', 'at', 'and', 'exclamation', 'space', 'tilde', 'comma', 'plus', 'asterisk', 'hashtag', 'dollar', 'percent']}
    
    return {
        'dot': text.count('.'), 'hyphen': text.count('-'), 'underline': text.count('_'),
        'slash': text.count('/'), 'questionmark': text.count('?'), 'equal': text.count('='),
        'at': text.count('@'), 'and': text.count('&'), 'exclamation': text.count('!'),
        'space': text.count(' '), 'tilde': text.count('~'), 'comma': text.count(','),
        'plus': text.count('+'), 'asterisk': text.count('*'), 'hashtag': text.count('#'),
        'dollar': text.count('$'), 'percent': text.count('%')
    }

def extract_111_features(url):
    # Standardize URL
    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url
    
    parsed = urlparse(url)
    
    domain = parsed.netloc
    path = parsed.path
    params = parsed.query
    
    # Determine what is missing (to apply the -1 rule)
    has_directory = '/' in path and len(path) > 1
    has_file = '.' in os.path.basename(path)
    has_params = len(params) > 0
    
    directory = os.path.dirname(path) if has_directory else ""
    file = os.path.basename(path) if has_file else ""
    
    # Extract counts (Passing True if missing to get -1s)
    u_counts = get_counts(url)
    d_counts = get_counts(domain)
    dir_counts = get_counts(directory, not has_directory)
    f_counts = get_counts(file, not has_file)
    p_counts = get_counts(params, not has_params)
    
    features = {}
    
    # Map counts to column names
    for char in u_counts:
        features[f'qty_{char}_url'] = u_counts[char]
        features[f'qty_{char}_domain'] = d_counts[char]
        features[f'qty_{char}_directory'] = dir_counts[char]
        features[f'qty_{char}_file'] = f_counts[char]
        features[f'qty_{char}_params'] = p_counts[char]
        
    # Add Lengths (Use -1 if missing)
    features['length_url'] = len(url)
    features['domain_length'] = len(domain)
    features['directory_length'] = len(directory) if has_directory else -1
    features['file_length'] = len(file) if has_file else -1
    features['params_length'] = len(params) if has_params else -1
    features['qty_params'] = len(params.split('&')) if has_params else -1
    
    # Special Features
    features['email_in_url'] = 1 if re.search(r'[\w\.-]+@[\w\.-]+', url) else 0
    features['tls_ssl_certificate'] = 1 if url.startswith('https') else 0
    
    # Fill remaining required columns with -1 to be safe
    final_row = [features.get(col, -1) for col in expected_features]
    return pd.DataFrame([final_row], columns=expected_features)

# --- LIVE TEST ---
print("--- C-SEC ALIGNED PREDICTOR ---")
while True:
    test_url = input("\nEnter URL to analyze: ").strip()
    if test_url.lower() == 'exit': break
    
    features_df = extract_111_features(test_url)
    prediction = model.predict(features_df)[0]
    confidence = model.predict_proba(features_df)[0]
    
    status = "🚩 PHISHING" if prediction == 1 else "✅ SAFE"
    conf_score = confidence[1] if prediction == 1 else confidence[0]
    
    print(f"Result: {status}")
    print(f"Confidence: {conf_score*100:.2f}%")

--- C-SEC ALIGNED PREDICTOR ---
Result: 🚩 PHISHING
Confidence: 73.00%
Result: 🚩 PHISHING
Confidence: 73.00%
Result: ✅ SAFE
Confidence: 88.00%
Result: 🚩 PHISHING
Confidence: 59.00%
Result: ✅ SAFE
Confidence: 79.00%
